In [ ]:
"""Quantify strand-specific mean coverage per CDS from GSE161360 WIG files.

WIG files are on genome build AE009951.2 (variableStep, span=1).
Coverage sign convention: forward files positive, reverse files negative
(verified during planning). We take absolute values and assign coverage to
genes by strand: forward-file coverage -> plus-strand genes; reverse-file
coverage -> minus-strand genes.

Outputs:
  coverage_mean.tsv   - mean coverage per CDS x sample
  rpkm.tsv            - RPKM-like normalized values
  sample_metadata.tsv - corrected sample metadata
  gene_annotation.tsv - CDS annotation from GenBank
"""
import gzip, os, re, glob
import numpy as np
import pandas as pd
from Bio import SeqIO

WIG_DIR = "."
GBK = "25586_NCBI.gb"
READ_LEN = 75
OUT = "."

# ---------- 1. Parse GenBank CDS annotation ----------
rec = SeqIO.read(GBK, "genbank")
genome_len = len(rec.seq)
print(f"Genome: {rec.id}, length {genome_len}")

genes = []
for feat in rec.features:
    if feat.type != "CDS":
        continue
    lt = feat.qualifiers.get("locus_tag", [""])[0]
    if len(feat.location.parts) > 1:
        # joined feature (e.g. FN1495 wraps the oriC) - skip; not in any PAI
        continue
    start = int(feat.location.start)  # 0-based
    end = int(feat.location.end)      # 1-based end
    strand = feat.location.strand             # +1 or -1
    prod = feat.qualifiers.get("product", [""])[0]
    gene_name = feat.qualifiers.get("gene", [""])[0]
    genes.append(dict(locus_tag=lt, start=start, end=end, strand=strand,
                      length=end - start, product=prod, gene=gene_name))
gdf = pd.DataFrame(genes).sort_values("start").reset_index(drop=True)
print(f"CDS features: {len(gdf)}")

# PAI membership
def pai_of(lt):
    m = re.match(r"FN(\d+)$", lt)
    if not m:
        return ""
    n = int(m.group(1))
    if 1878 <= n <= 1887: return "PAI1"
    if 834 <= n <= 865:  return "PAI2"
    if 1313 <= n <= 1345: return "PAI3"
    return ""
gdf["PAI"] = gdf["locus_tag"].map(pai_of)
print(gdf["PAI"].value_counts())
gdf.to_csv(f"{OUT}/gene_annotation.tsv", sep="\t", index=False)

# ---------- 2. Sample metadata ----------
files = sorted(glob.glob(f"{WIG_DIR}/*.wig.gz"))
rows = []
for f in files:
    base = os.path.basename(f)
    m = re.match(r"(GSM\d+)_(ID-\d+)_FNN_([EMS])_(untreated|tex_plus)_rep([ABC])_(forward|reverse)\.wig\.gz", base)
    gsm, idn, phase, treat, rep, strand_dir = m.groups()
    rows.append(dict(gsm=gsm, idn=idn, phase=phase, treatment=treat,
                     replicate=rep, direction=strand_dir, file=base))
sdf = pd.DataFrame(rows)
samples = (sdf[["gsm", "idn", "phase", "treatment", "replicate"]]
           .drop_duplicates().sort_values("gsm").reset_index(drop=True))
samples.to_csv(f"{OUT}/sample_metadata.tsv", sep="\t", index=False)
print(f"Samples: {len(samples)}")
print(samples.groupby(["treatment", "phase"]).size())

# ---------- 3. Parse WIGs -> per-base coverage arrays ----------
def parse_wig(path, genome_len):
    cov = np.zeros(genome_len, dtype=np.float64)
    chrom, pos = None, None
    with gzip.open(path, "rt") as fh:
        for line in fh:
            if line.startswith("variableStep"):
                chrom = line.split()[1].split("=")[1]
                continue
            if line.startswith("track") or not line.strip():
                continue
            p, v = line.split()
            cov[int(p) - 1] = float(v)
    return cov

# ---------- 4. Quantify ----------
sample_ids = samples["gsm"].tolist()
mean_cov = pd.DataFrame(index=gdf["locus_tag"], columns=sample_ids, dtype=float)
lib_sizes = {}

for gsm in sample_ids:
    fwd_file = sdf[(sdf.gsm == gsm) & (sdf.direction == "forward")]["file"].iloc[0]
    rev_file = sdf[(sdf.gsm == gsm) & (sdf.direction == "reverse")]["file"].iloc[0]
    fwd = parse_wig(f"{WIG_DIR}/{fwd_file}", genome_len)
    rev = parse_wig(f"{WIG_DIR}/{rev_file}", genome_len)
    # library size proxy: total mapped bases / read length
    lib_sizes[gsm] = (np.abs(fwd).sum() + np.abs(rev).sum()) / READ_LEN
    vals = []
    for _, g in gdf.iterrows():
        arr = fwd if g["strand"] == 1 else rev
        vals.append(np.abs(arr[g["start"]:g["end"]]).mean())
    mean_cov[gsm] = vals
    print(f"{gsm}: lib_size~{lib_sizes[gsm]/1e6:.2f}M reads")

mean_cov.to_csv(f"{OUT}/coverage_mean.tsv", sep="\t")

# ---------- 5. RPKM-like normalization ----------
lib = pd.Series(lib_sizes)  # approx mapped reads
rpkm = mean_cov.div(lib / 1e6, axis=1)  # mean_cov per million mapped reads
rpkm.to_csv(f"{OUT}/rpkm.tsv", sep="\t")
print("RPKM summary:")
print(rpkm.describe().loc[["mean", "50%", "max"]].round(2))
print("Done.")


Genome: AE009951.2, length 2174500
CDS features: 2066
PAI
        1991
PAI3      33
PAI2      32
PAI1      10
Name: count, dtype: int64
Samples: 18
treatment  phase
tex_plus   E        3
           M        3
           S        3
untreated  E        3
           M        3
           S        3
dtype: int64
GSM4905285: lib_size~8.83M reads
GSM4905286: lib_size~8.07M reads
GSM4905287: lib_size~8.20M reads
GSM4905288: lib_size~9.18M reads
GSM4905289: lib_size~8.98M reads
GSM4905290: lib_size~8.27M reads
GSM4905291: lib_size~8.03M reads
GSM4905292: lib_size~8.61M reads
GSM4905293: lib_size~8.97M reads
GSM4905294: lib_size~7.10M reads
GSM4905295: lib_size~7.10M reads
GSM4905296: lib_size~7.63M reads
GSM4905297: lib_size~9.59M reads
GSM4905298: lib_size~8.03M reads
GSM4905299: lib_size~7.91M reads
GSM4905300: lib_size~7.41M reads
GSM4905301: lib_size~8.03M reads
GSM4905302: lib_size~8.35M reads
RPKM summary:
      GSM4905285  GSM4905286  GSM4905287  GSM4905288  GSM4905289  GSM4905290  \
me

In [5]:
# ═══════════════════════════════════════════════════════════════════════
# Check if Prokka annotations matches the RNA-seq data
# ═══════════════════════════════════════════════════════════════════════
WIGDIR = "."          # directory with your 36 *FNN*.wig.gz files
GFF    = "25586.gff"      # Prokka GFF annotation
FNA    = "25586.fna"      # genome FASTA
REFGB  = "25586_NCBI.gb"  # NCBI reference GenBank (for boundary comparison)
OUTDIR = "."      # output directory
# also need the zip file for the rna seq
# ═══════════════════════════════════════════════════════════════════════

import argparse, gzip, os, re, sys, csv, statistics
from collections import defaultdict

import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'
import matplotlib.pyplot as plt

# Colorblind-safe palette (Okabe-Ito)
BLUE   = '#0072B2'
ORANGE = '#E69F00'
GREY   = '#999999'

GENOME_LEN = 2174500  # AE009951.2


# ═══════════════════════════════════════════════════════════════════════
# 1. PARSE PROKKA GFF  (the annotation we are validating)
# ═══════════════════════════════════════════════════════════════════════
def parse_prokka_gff(path):
    """Return list of dicts: locus_tag, start(1-based), end, strand, gene, product."""
    cds = []
    with open(path) as fh:
        for line in fh:
            if line.startswith('#'):
                continue
            f = line.rstrip('\n').split('\t')
            if len(f) < 9 or f[2] != 'CDS':
                continue
            attrs = f[8]
            lt   = _attr(attrs, 'locus_tag')
            prod = _attr(attrs, 'product')
            gene = _attr(attrs, 'gene')
            cds.append(dict(
                locus_tag=lt, start=int(f[3]), end=int(f[4]),
                strand=f[6], gene=gene or '', product=prod or '',
            ))
    return cds

def _attr(attrs_str, key):
    m = re.search(r'(?:^|;)' + re.escape(key) + r'=([^;]*)', attrs_str)
    return m.group(1) if m else ''


# ═══════════════════════════════════════════════════════════════════════
# 2. PARSE NCBI REFERENCE GENBANK  (for boundary comparison only)
# ═══════════════════════════════════════════════════════════════════════
def parse_ref_genbank(path):
    """Return list of dicts: locus_tag, start, end, strand, product."""
    cds = []
    with open(path) as fh:
        text = fh.read()
    # Split into CDS feature blocks
    blocks = re.findall(r'\n     CDS             ([^\n]+)\n((?:                     /[^\n]+\n?)+)', text)
    for loc, quals in blocks:
        loc = loc.strip()
        strand = '-' if loc.startswith('complement') else '+'
        nums = [int(x) for x in re.findall(r'\d+', loc)]
        if not nums:
            continue
        start, end = min(nums), max(nums)
        lt   = re.search(r'/locus_tag="([^"]+)"', quals)
        prod = re.search(r'/product="([^"]+)"', quals)
        cds.append(dict(
            locus_tag=lt.group(1) if lt else '?',
            start=start, end=end, strand=strand,
            product=prod.group(1) if prod else '?',
        ))
    return cds


# ═══════════════════════════════════════════════════════════════════════
# 3. BOUNDARY COMPARISON  (Prokka vs NCBI reference)
# ═══════════════════════════════════════════════════════════════════════
def boundary_comparison(prokka_cds, ref_cds):
    ref_by_pos = {(c['start'], c['end'], c['strand']): c for c in ref_cds}
    ref_starts = defaultdict(list)
    ref_ends   = defaultdict(list)
    for c in ref_cds:
        ref_starts[(c['start'], c['strand'])].append(c)
        ref_ends[(c['end'], c['strand'])].append(c)

    ref_intervals = [(c['start'], c['end'], c['strand']) for c in ref_cds]
    def overlaps_ref(s, e, st):
        for rs, re_, rs_ in ref_intervals:
            if rs_ == st and s <= re_ and rs <= e:
                return True
        return False

    matched_ref = set()
    cats = defaultdict(list)
    for p in prokka_cds:
        s, e, st = p['start'], p['end'], p['strand']
        if (s, e, st) in ref_by_pos:
            cat = 'exact_match'
        elif (s, st) in ref_starts:
            cat = 'same_start_diff_end'
        elif (e, st) in ref_ends:
            cat = 'same_end_diff_start'
        elif overlaps_ref(s, e, st):
            cat = 'other_overlap'
        else:
            cat = 'prokka_only'
        cats[cat].append(p)

    for i, (rs, re_, rs_) in enumerate(ref_intervals):
        for p in prokka_cds:
            if p['strand'] == rs_ and p['start'] <= re_ and rs <= p['end']:
                matched_ref.add(i)
                break
    cats['ref_only'] = [ref_cds[i] for i in range(len(ref_cds)) if i not in matched_ref]
    return cats


# ═══════════════════════════════════════════════════════════════════════
# 4. PARSE WIG FILES
# ═══════════════════════════════════════════════════════════════════════
def parse_wig(path, genome_len):
    """Parse a variableStep WIG file (gzipped) into a float32 coverage array."""
    cov = np.zeros(genome_len, dtype=np.float32)
    with gzip.open(path, 'rt') as fh:
        for line in fh:
            if line.startswith('track') or line.startswith('variableStep') or not line.strip():
                continue
            parts = line.split()
            if len(parts) == 2:
                pos = int(parts[0]) - 1  # WIG is 1-based; array is 0-based
                val = abs(float(parts[1]))  # reverse strand has negative values
                if 0 <= pos < genome_len:
                    cov[pos] = val
    return cov


def discover_samples(wigdir):
    """Find FNN forward/reverse WIG pairs and return sample metadata."""
    pattern = re.compile(
        r'(GSM\d+)_(ID-\d+)_FNN_([EMS])_(untreated|tex_plus)_rep([ABC])_(forward|reverse)\.wig\.gz'
    )
    samples = {}
    for fname in sorted(os.listdir(wigdir)):
        m = pattern.match(fname)
        if not m:
            continue
        gsm, idn, phase, treat, rep, direction = m.groups()
        key = gsm
        if key not in samples:
            samples[key] = dict(gsm=gsm, idn=idn, phase=phase, treatment=treat,
                                replicate=rep, forward=None, reverse=None)
        samples[key][direction] = fname
    # keep only samples with both forward and reverse
    complete = {k: v for k, v in samples.items() if v['forward'] and v['reverse']}
    return dict(sorted(complete.items()))


# ═══════════════════════════════════════════════════════════════════════
# 5. COMPUTE CONCORDANCE METRICS
# ═══════════════════════════════════════════════════════════════════════
def compute_metrics(samples, prokka_cds, genome_len, outdir):
    """
    Process each sample: parse WIGs, compute per-gene mean coverage,
    CDS/intergenic coverage, strand concordance.
    Returns: per_gene (dict locus_tag -> list of mean_cov per sample),
             per_sample (list of dicts), sample_ids (list)
    """
    sample_ids = list(samples.keys())
    n_cds = len(prokka_cds)

    # Precompute CDS intervals (0-based for array indexing)
    cds_intervals = []
    for c in prokka_cds:
        s0 = c['start'] - 1  # 0-based start
        e0 = c['end']        # 0-based exclusive end
        cds_intervals.append((s0, e0, c['strand'], c['locus_tag']))

    # Precompute merged annotated intervals (CDS + rRNA + tRNA + tmRNA from GFF)
    # and intergenic intervals for coverage / gap scan
    # (We re-parse the GFF for all feature types)
    # Done by caller via build_intervals()

    per_gene = {c['locus_tag']: [] for c in prokka_cds}
    per_sample = []

    for sid in sample_ids:
        meta = samples[sid]
        fwd_path = os.path.join(args.wigdir, meta['forward'])
        rev_path = os.path.join(args.wigdir, meta['reverse'])

        try:
            fwd = parse_wig(fwd_path, genome_len)
            rev = parse_wig(rev_path, genome_len)
        except Exception as e:
            print(f"  WARNING: failed to parse {sid}: {e}", file=sys.stderr)
            per_sample.append(dict(sample=sid, phase=meta['phase'],
                                   treatment=meta['treatment'],
                                   replicate=meta['replicate'],
                                   genome_cov_pct=np.nan, cds_cov_pct=np.nan,
                                   intergenic_cov_pct=np.nan,
                                   strand_concordance_pct=np.nan,
                                   mean_cds_coverage=np.nan, status='FAILED'))
            for c in prokka_cds:
                per_gene[c['locus_tag']].append(np.nan)
            continue

        total_cov = fwd + rev  # combined coverage for genome-wide stats

        # --- per-gene mean coverage from correct strand ---
        for s0, e0, strand, lt in cds_intervals:
            arr = fwd if strand == '+' else rev
            mc = float(np.abs(arr[s0:e0]).mean())
            per_gene[lt].append(mc)

        # --- genome-wide coverage ---
        genome_cov_pct = np.count_nonzero(total_cov) / genome_len * 100

        # --- CDS coverage ---
        cds_mask = np.zeros(genome_len, dtype=bool)
        for s0, e0, strand, lt in cds_intervals:
            cds_mask[s0:e0] = True
        cds_cov_pct = np.count_nonzero(total_cov[cds_mask]) / np.count_nonzero(cds_mask) * 100

        # --- intergenic coverage ---
        inter_mask = ~cds_mask
        inter_cov_pct = np.count_nonzero(total_cov[inter_mask]) / np.count_nonzero(inter_mask) * 100

        # --- strand concordance per CDS ---
        concordances = []
        for s0, e0, strand, lt in cds_intervals:
            correct = fwd if strand == '+' else rev
            wrong   = rev if strand == '+' else fwd
            c_sum = float(np.abs(correct[s0:e0]).sum())
            w_sum = float(np.abs(wrong[s0:e0]).sum())
            if c_sum + w_sum > 0:
                concordances.append(c_sum / (c_sum + w_sum) * 100)
        strand_conc = np.median(concordances) if concordances else np.nan

        mean_cds_cov = float(np.mean([per_gene[c['locus_tag']][-1] for c in prokka_cds]))

        per_sample.append(dict(
            sample=sid, phase=meta['phase'], treatment=meta['treatment'],
            replicate=meta['replicate'], genome_cov_pct=genome_cov_pct,
            cds_cov_pct=cds_cov_pct, intergenic_cov_pct=inter_cov_pct,
            strand_concordance_pct=strand_conc, mean_cds_coverage=mean_cds_cov,
            status='OK',
        ))
        print(f"  {sid} ({meta['phase']}.{meta['treatment']}.{meta['replicate']}): "
              f"genome={genome_cov_pct:.1f}% CDS={cds_cov_pct:.1f}% "
              f"inter={inter_cov_pct:.1f}% strand={strand_conc:.2f}%")

    return per_gene, per_sample, sample_ids


def build_intervals(gff_path, genome_len):
    """Build merged annotated intervals and intergenic intervals from GFF."""
    feats = []
    with open(gff_path) as fh:
        for line in fh:
            if line.startswith('#'):
                continue
            f = line.rstrip('\n').split('\t')
            if len(f) < 9 or f[2] not in ('CDS', 'rRNA', 'tRNA', 'tmRNA'):
                continue
            feats.append((int(f[3]), int(f[4])))
    feats.sort()
    merged = []
    for s, e in feats:
        if merged and s <= merged[-1][1] + 1:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    # intergenic = complement of merged
    intergenic = []
    prev = 0
    for s, e in merged:
        if s - 1 > prev:
            intergenic.append((prev + 1, s - 1))
        prev = max(prev, e)
    if prev < genome_len:
        intergenic.append((prev + 1, genome_len))
    return merged, intergenic


def gap_scan(samples, intergenic, genome_len, outdir):
    """Scan intergenic regions for high coverage (candidate missed genes)."""
    sample_ids = list(samples.keys())
    # For each intergenic region, compute mean coverage per sample from both strands
    results = []
    for sid in sample_ids:
        meta = samples[sid]
        fwd = parse_wig(os.path.join(args.wigdir, meta['forward']), genome_len)
        rev = parse_wig(os.path.join(args.wigdir, meta['reverse']), genome_len)
        total = fwd + rev
        for s, e in intergenic:
            length = e - s + 1
            if length < 100:
                continue
            mc = float(total[s-1:e].mean())
            results.append(dict(sample=sid, start=s, end=e, length=length, mean_cov=mc))

    # Aggregate: for each intergenic region, count samples with mean_cov > 5
    from collections import defaultdict
    region_stats = defaultdict(list)
    for r in results:
        region_stats[(r['start'], r['end'], r['length'])].append(r['mean_cov'])

    candidates = []
    for (s, e, length), covs in region_stats.items():
        n_high = sum(1 for c in covs if c > 5)
        if n_high >= 3:
            candidates.append(dict(start=s, end=e, length=length,
                                   n_samples_cov_gt5=n_high,
                                   max_mean_cov=max(covs),
                                   mean_cov=np.mean(covs)))
    return candidates


# ═══════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════
def main(args):
    os.makedirs(args.outdir, exist_ok=True)

    # --- genome length ---
    genome_len = 0
    with open(args.fna) as fh:
        for line in fh:
            if not line.startswith('>'):
                genome_len += len(line.strip())
    print(f"Genome length: {genome_len}")

    # --- 1. Parse Prokka GFF ---
    prokka_cds = parse_prokka_gff(args.gff)
    print(f"Prokka CDS: {len(prokka_cds)}")
    n_hyp = sum(1 for c in prokka_cds if 'hypothetical' in c['product'].lower())
    print(f"  hypothetical: {n_hyp} ({n_hyp/len(prokka_cds)*100:.1f}%)")

    # --- 2. Parse NCBI reference GenBank ---
    ref_cds = parse_ref_genbank(args.refgb)
    print(f"NCBI reference CDS: {len(ref_cds)}")

    # --- 3. Boundary comparison ---
    print("\n=== Boundary comparison (Prokka vs NCBI reference) ===")
    bcats = boundary_comparison(prokka_cds, ref_cds)
    for cat in ['exact_match', 'same_start_diff_end', 'same_end_diff_start',
                'other_overlap', 'prokka_only', 'ref_only']:
        print(f"  {cat}: {len(bcats.get(cat, []))}")

    # Write boundary comparison table
    with open(os.path.join(args.outdir, 'boundary_comparison.tsv'), 'w', newline='') as f:
        w = csv.writer(f, delimiter='\t')
        w.writerow(['category', 'locus_tag', 'start', 'end', 'strand', 'product'])
        for cat in ['exact_match', 'same_start_diff_end', 'same_end_diff_start',
                    'other_overlap', 'prokka_only', 'ref_only']:
            for c in bcats.get(cat, []):
                w.writerow([cat, c['locus_tag'], c['start'], c['end'],
                            c.get('strand', '?'), c.get('product', '?')])

    # --- 4. Discover WIG samples ---
    print("\n=== Discovering FNN WIG samples ===")
    samples = discover_samples(args.wigdir)
    print(f"Found {len(samples)} samples (forward + reverse pairs)")
    if len(samples) == 0:
        print("ERROR: No FNN WIG files found. Expected files matching "
              "*FNN_*_forward.wig.gz and *FNN_*_reverse.wig.gz", file=sys.stderr)
        sys.exit(1)

    # --- 5. Compute concordance metrics ---
    print("\n=== Computing concordance metrics ===")
    per_gene, per_sample, sample_ids = compute_metrics(
        samples, prokka_cds, genome_len, args.outdir)

    # --- 6. Write per-gene coverage matrix ---
    with open(os.path.join(args.outdir, 'per_gene_coverage.tsv'), 'w', newline='') as f:
        w = csv.writer(f, delimiter='\t')
        w.writerow(['locus_tag', 'strand', 'product'] + sample_ids)
        for c in prokka_cds:
            vals = per_gene[c['locus_tag']]
            w.writerow([c['locus_tag'], c['strand'], c['product']] +
                       [f'{v:.4f}' if not np.isnan(v) else 'NA' for v in vals])

    # --- 7. Write per-sample stats ---
    with open(os.path.join(args.outdir, 'per_sample_stats.tsv'), 'w', newline='') as f:
        w = csv.writer(f, delimiter='\t')
        w.writerow(['sample', 'phase', 'treatment', 'replicate', 'genome_cov_pct',
                    'cds_cov_pct', 'intergenic_cov_pct', 'strand_concordance_pct',
                    'mean_cds_coverage', 'status'])
        for s in per_sample:
            w.writerow([s['sample'], s['phase'], s['treatment'], s['replicate'],
                        f"{s['genome_cov_pct']:.2f}" if s['status'] == 'OK' else 'NA',
                        f"{s['cds_cov_pct']:.2f}" if s['status'] == 'OK' else 'NA',
                        f"{s['intergenic_cov_pct']:.2f}" if s['status'] == 'OK' else 'NA',
                        f"{s['strand_concordance_pct']:.2f}" if s['status'] == 'OK' else 'NA',
                        f"{s['mean_cds_coverage']:.4f}" if s['status'] == 'OK' else 'NA',
                        s['status']])

    # --- 8. Write unexpressed annotated genes ---
    n_samples = len(sample_ids)
    ann = [c for c in prokka_cds if 'hypothetical' not in c['product'].lower()]
    unexpr = []
    for c in ann:
        n_expr = sum(1 for i in range(n_samples)
                     if not np.isnan(per_gene[c['locus_tag']][i])
                     and not np.isnan(per_gene[c['locus_tag']][i])
                     and per_gene[c['locus_tag']][i] > 1)
        if n_expr < 3:
            max_cov = max((per_gene[c['locus_tag']][i] for i in range(n_samples)
                           if not np.isnan(per_gene[c['locus_tag']][i])
                           and not np.isnan(per_gene[c['locus_tag']][i])),
                          default=0)
            unexpr.append((c['locus_tag'], c['product'], f'{max_cov:.4f}'))
    with open(os.path.join(args.outdir, 'unexpressed_annotated.tsv'), 'w', newline='') as f:
        w = csv.writer(f, delimiter='\t')
        w.writerow(['locus_tag', 'product', 'max_mean_coverage'])
        w.writerows(unexpr)

    # --- 9. Gap scan ---
    print("\n=== Gap scan (expressed intergenic regions) ===")
    merged, intergenic = build_intervals(args.gff, genome_len)
    gap_candidates = gap_scan(samples, intergenic, genome_len, args.outdir)
    print(f"  Expressed intergenic regions (>100bp, mean cov>5 in >=3 samples): {len(gap_candidates)}")

    with open(os.path.join(args.outdir, 'expressed_intergenic.tsv'), 'w', newline='') as f:
        w = csv.writer(f, delimiter='\t')
        w.writerow(['start', 'end', 'length', 'n_samples_cov_gt5', 'max_mean_cov', 'mean_cov'])
        for g in sorted(gap_candidates, key=lambda x: -x['max_mean_cov']):
            w.writerow([g['start'], g['end'], g['length'], g['n_samples_cov_gt5'],
                        f"{g['max_mean_cov']:.2f}", f"{g['mean_cov']:.2f}"])



    # --- Summary ---
    ok = [s for s in per_sample if s['status'] == 'OK']
    print(f"\n{'='*60}")
    print(f"DONE. {len(ok)}/{len(sample_ids)} samples processed successfully.")
    print(f"Outputs in: {args.outdir}")
    print(f"{'='*60}")



# ═══════════════════════════════════════════════════════════════════════
# RUN
# ═══════════════════════════════════════════════════════════════════════
args = argparse.Namespace(wigdir=WIGDIR, gff=GFF, fna=FNA, refgb=REFGB, outdir=OUTDIR)
main(args)


Genome length: 2174500
Prokka CDS: 2016
  hypothetical: 875 (43.4%)
NCBI reference CDS: 2067

=== Boundary comparison (Prokka vs NCBI reference) ===
  exact_match: 1415
  same_start_diff_end: 307
  same_end_diff_start: 272
  other_overlap: 11
  prokka_only: 11
  ref_only: 63

=== Discovering FNN WIG samples ===
Found 18 samples (forward + reverse pairs)

=== Computing concordance metrics ===
  GSM4905285 (E.untreated.A): genome=82.2% CDS=83.7% inter=67.9% strand=100.00%
  GSM4905286 (E.untreated.B): genome=82.3% CDS=83.7% inter=69.1% strand=100.00%
  GSM4905287 (E.untreated.C): genome=80.3% CDS=81.8% inter=65.8% strand=100.00%
  GSM4905288 (M.untreated.A): genome=83.6% CDS=85.2% inter=68.5% strand=100.00%
  GSM4905289 (M.untreated.B): genome=83.5% CDS=85.0% inter=68.3% strand=100.00%
  GSM4905290 (M.untreated.C): genome=84.9% CDS=86.4% inter=70.1% strand=100.00%
  GSM4905291 (S.untreated.A): genome=77.6% CDS=78.8% inter=65.7% strand=100.00%
  GSM4905292 (S.untreated.B): genome=83.4% CD